In [2]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from statsmodels.stats.proportion import proportion_confint
import numpy as np
con=sqlite3.connect('C:\\Users\\yasee\\OneDrive\\Desktop\\Sem 3 projects\\project 3.1\\data\\raw\\ipl.db')

def q(sql):
    return pd.read_sql(sql,con)

In [3]:
stats.binomtest(613, 1187, 0.5).pvalue

np.float64(0.2700389712415702)

In [4]:
#Does winning the toss help you win the match ?
q("""SELECT COUNT (*) AS n,
      SUM(CASE WHEN  toss_winner=match_winner
              THEn 1 ELSE 0 END) AS k
              FROM matches_clean
              WHERE result = 'win';""")

,n,k
0,1187,613


In [5]:
# syntax :stats.binomtest(k, n, p)
#k=no of success
#n=total trail,data
#p=expected probability
p=stats.binomtest(613, 1187,0.5).pvalue
ci=proportion_confint(613, 1187, method='wilson')
print(p)
print(ci)
#pvalue<significant level THEN the h0 is significant 
#pvalue>significant level THEN the h0 is not  significant 

0.2700389712415702
(0.48799203833275506, 0.5447579131499739)


In [6]:
# ho-The toss winner wins half the time. The 51.64% we saw is luck, and nothing more.
#h1 -The toss winner does not win half the time
6

6

In [7]:
#Does the decision taken after the toss matter?
#h0- Captains who choose to bat and captains who 
# choose to field win equally often. The gap we saw is luck.
#h1-The two choices do not win equally often.

In [8]:
q("""SELECT toss_decision,
       SUM(CASE WHEN toss_winner = match_winner
           THEN 1 ELSE 0 END) AS won,
       SUM(CASE WHEN toss_winner <> match_winner
           THEN 1 ELSE 0 END) AS lost
FROM   matches_clean
WHERE  result = 'win'
GROUP BY toss_decision;""")

,toss_decision,won,lost
0,bat,184,218
1,field,429,356


In [9]:
table = [[184, 218],  # chose to bat: won, lost
         [429, 356]]  # chose to field: won, lost
chi2, p, dof, expected = stats.chi2_contingency(table)
print(round(chi2, 2), round(p, 4))

8.04 0.0046


In [10]:
#Does the chasing team really have an advantage?
#h0-Both sides win half the time. The 54.51% we saw is luck.
#h1-A chase does not succeed half the time.
q(""" SELECT COUNT(*) AS n, 
SUM(chase_won)AS k 
FROM v_match_totals;""")

,n,k
0,1187,647


In [11]:
#n1- The chase win rate is higher
q("""SELECT CASE WHEN first_innings_runs < 140 THEN 'Under 140'
            WHEN first_innings_runs < 160 THEN '140-159'
            WHEN first_innings_runs < 180 THEN '160-179'
            WHEN first_innings_runs < 200 THEN '180-199'
            ELSE '200+' END AS band,
       SUM(chase_won)     AS won,
       SUM(1 - chase_won) AS lost
FROM   v_match_totals
GROUP  BY band;""")

,band,won,lost
0,140-159,171,70
1,160-179,166,146
2,180-199,83,134
3,200+,46,152
4,Under 140,181,38


In [12]:
table = [[181, 38],
         [171, 70],
         [166,146],
         [83,134],
         [46,152]]
chi2, p, dof, expected = stats.chi2_contingency(table)

print(round(chi2, 2), round(p, 4))

197.68 0.0


In [13]:
#Are defended totals actually bigger than chased ones?

#h0=Both kinds of total average the same. The gap we see is luck
#h1=The two averages are not equal.

q("""SELECT first_innings_runs, chase_won
FROM   v_match_totals;""")

,first_innings_runs,chase_won
0,222,0
1,207,0
2,184,1
3,183,1
4,163,1
...,...,...
1182,75,1
1183,222,1
1184,243,1
1185,155,1


In [14]:
df=pd.read_sql("""SELECT first_innings_runs, chase_won
FROM   v_match_totals;""",con)
defended = df[df.chase_won == 0].first_innings_runs
chased   = df[df.chase_won == 1].first_innings_runs
t, p = stats.ttest_ind(defended, chased, equal_var=False)
# move 6 needs the effect size too
pooled = np.sqrt((defended.std()**2 + chased.std()**2) / 2)
d = (defended.mean() - chased.mean()) / pooled
print(d)
print(pooled)
print(t)
print(p)

0.9162898870801979
30.342879534020167
15.730238965353694
1.17272268263676e-50


In [15]:
df=pd.read_sql("""SELECT match_id, innings,
       6.0*SUM(CASE WHEN phase='Powerplay' THEN total_runs ELSE 0 END)
         / SUM(CASE WHEN phase='Powerplay' THEN is_legal ELSE 0 END) AS pp,
       6.0*SUM(CASE WHEN phase='Middle' THEN total_runs ELSE 0 END)
         / SUM(CASE WHEN phase='Middle' THEN is_legal ELSE 0 END) AS mid
FROM   v_ball
WHERE  innings IN (1,2)
GROUP  BY match_id, innings;""",con)
df



,match_id,innings,pp,mid
0,335982,1,10.166667,10.333333
1,335982,2,4.333333,6.111111
2,335983,1,8.833333,12.000000
3,335983,2,10.500000,11.333333
4,335984,1,6.666667,6.222222
...,...,...,...,...
2413,1529284,2,15.333333,11.222222
2414,1529285,1,9.833333,8.111111
2415,1529285,2,11.500000,8.444444
2416,1529286,1,9.333333,10.111111


In [16]:
df=pd.read_sql("""SELECT Match_id, innings,
       6.0 * SUM(CASE WHEN phase='Powerplay' THEN total_runs ELSE 0 END)
       / SUM(CASE WHEN phase='Powerplay' THEN is_legal ELSE 0 END) AS pp,

       6.0 * SUM(CASE WHEN phase='Middle' THEN total_runs ELSE 0 END)
       / SUM(CASE WHEN phase='Middle' THEN is_legal ELSE 0 END) AS mid

FROM v_ball

WHERE innings IN (1,2)

GROUP BY match_id, innings

HAVING
    SUM(CASE WHEN phase='Powerplay' THEN is_legal ELSE 0 END) > 0
    AND
    SUM(CASE WHEN phase='Middle' THEN is_legal ELSE 0 END) > 0;""",con)
df

,match_id,innings,pp,mid
0,335982,1,10.166667,10.333333
1,335982,2,4.333333,6.111111
2,335983,1,8.833333,12.000000
3,335983,2,10.500000,11.333333
4,335984,1,6.666667,6.222222
...,...,...,...,...
2401,1529284,2,15.333333,11.222222
2402,1529285,1,9.833333,8.111111
2403,1529285,2,11.500000,8.444444
2404,1529286,1,9.333333,10.111111


In [17]:
t,p = stats.ttest_rel(df.pp, df.mid)
print(round(t, 2), 
print(round(p, 4)))
print(p<0.05)

0.1259
1.53 None
False


In [18]:
#Do left-handers really hit harder than right-handers?
#claim:Left-handed batters have a higher strike rate than right-handed batters.
#h0=Both hit at the same rate. The gap found is luck.
# h1=The two average strike rates are not equal.

df=pd.read_sql("""
SELECT batter, batsman_type,
    100.0*SUM(batter_runs)/SUM(is_legal) AS sr,
    SUM(is_legal) AS balls
FROM v_ball
WHERE batsman_type IS NOT NULL
GROUP BY batter, batsman_type
HAVING Balls>=500
;""",con)
df

,batter,batsman_type,sr,balls
0,A Badoni,Right hand Bat,139.263804,815
1,A Symonds,Right hand Bat,130.738255,745
2,AB de Villiers,Right hand Bat,152.181604,3392
3,AC Gilchrist,Left hand Bat,139.514498,1483
4,AD Mathews,Right hand Bat,126.352531,573
...,...,...,...,...
132,Washington Sundar,Left hand Bat,130.518234,521
133,Y Venugopal Rao,Right hand Bat,118.389423,832
134,YBK Jaiswal,Left hand Bat,154.104478,1608
135,YK Pathan,Right hand Bat,143.741588,2229


In [19]:
left = df[df.batsman_type == 'Left hand Bat']
right = df[df.batsman_type == 'Right hand Bat']
t,P=stats.ttest_ind(left.sr, right.sr,equal_var=False)
print(t)
print(P)
print(P < 0.05)


-0.007335404976981389
0.994160816878741
False


In [20]:
#Are spinners genuinely cheaper than fast bowlers?
# CLAIM:Spin bowlers concede fewer runs per over than fast bowlers
# h0=Both styles concede the same. The gap found is luck.
# h1=The two average economy rates are not equal.
df=pd.read_sql("""
        SELECT bowler,
    CASE WHEN bowler_type LIKE '%Legbreak%'
        OR bowler_type LIKE '%Offbreak%'
        OR bowler_type LIKE '%Orthodox%'
        OR bowler_type LIKE '%Googly%'
        OR bowler_type LIKE '%Wrist spin%'
        THEN 'Spin' ELSE 'Pace' END AS style,
    6.0*SUM(bowler_runs)/SUM(is_legal) AS economy,
    SUM(is_legal) AS balls
FROM v_ball
WHERE TRIM(COALESCE(bowler_type, '')) <> ''
GROUP BY bowler, style
HAVING Balls>=500;""",con)
df

,bowler,style,economy,balls
0,A Kumble,Spin,6.578238,965
1,A Mishra,Spin,7.377633,3371
2,A Nehra,Pace,7.845912,1908
3,A Nortje,Pace,9.085714,1120
4,A Symonds,Spin,7.673624,527
...,...,...,...,...
144,YK Pathan,Spin,7.401918,1147
145,YS Chahal,Spin,8.027912,3941
146,Yash Dayal,Pace,9.575201,871
147,Yuvraj Singh,Spin,7.436133,869


In [21]:
spin= df[df['style'] == 'Spin']
pace= df[df['style'] == 'Pace']
t,p=stats.ttest_ind(spin.economy, pace.economy, equal_var=False)
print(t)
print(p)
print(p<0.05)

pooled = np.sqrt((spin.economy.std()**2 + pace.economy.std()**2) / 2)
d = (pace.economy.mean() - spin.economy.mean()) / pooled
#0.2- small change
#0.5- medium change
#0.8- large change/effect  
#d=effect size--means how big the change is (cohens)
print(d)


-7.075597311698715
1.1249651433007181e-10
True
1.1866831126238373


In [22]:
# Does Sawai Mansingh really favour the chasing team?
# Chasing works better at Sawai Mansingh Stadium than it does elsewhere.

# h0= Chasing at Sawai Mansingh works exactly as well as chasing anywhere else— 54.51% 
# — and the 65.15% we saw is luck.
# h1=The chase rate at this ground is not 54.51%.
df=pd.read_sql("""SELECT count(*)  AS n,
         sum(chase_won) AS k
FROM   v_match_totals
WHERE  venue_clean = 'Sawai Mansingh Stadium';""",con)
df

,n,k
0,66,43


In [23]:
#Has scoring genuinely risen across the seasons?
#Claim:Average innings scores have risen over the nineteen seasons.
#h0-Average scores wander up and down with no trend, and the climb we saw is luck.
#h1-The season and the average score are related.
df=pd.read_sql("""SELECT m.season_year, AVG(i.runs) AS avg_score
FROM matches_clean m
JOIN v_innings i ON i.match_id = m.match_id
GROUP BY m.season_year;""",con)
df

,season_year,avg_score
0,2008,154.629310
1,2009,143.157895
2,2010,157.200000
3,2011,145.889655
4,2012,151.709459
5,2013,148.296053
6,2014,157.575000
7,2015,156.683761
8,2016,157.183333
9,2017,159.059322


In [24]:
#correlation 
#-1 to +1
# -1 =very strong-ve relation
#0 =no relation
# +1 =very strong+ve relation
r,p=stats.pearsonr(df.season_year, df.avg_score)
r_squared = r**2
print(round(r,3))
print(p)
print(round(r_squared,3))
print(p<0.05)


0.862
2.079722060152857e-06
0.743
True


In [26]:
# Is Mumbai's head-to-head record over Chennai real?
# Claim :Mumbai Indians have  the better of Chennai Super Kings.
# h0-The two teams are evenly matched, and the 21-19 record is luck.
# h1-Mumbai do not win half of the meetings.
df=pd.read_sql("""SELECT COUNT(*) AS n,
       SUM(CASE WHEN match_winner = 'Mumbai Indians' THEN 1 ELSE 0 END) AS k
FROM   matches_clean
WHERE result = 'Win'
AND   (team1 = 'Mumbai Indians' OR team2 = 'Chennai Super Kings')
OR (team1 = 'Chennai Super Kings' OR team2 = 'Mumbai Indians');""",con)

In [27]:
p=stats.binomtest(21, 40, 0.5).pvalue
ci=proportion_confint(21, 40, method='wilson')
print(p)
print(ci)
print(p<0.05)

0.8746293123804207
(0.37497362106692467, 0.6706452988732111)
False


In [28]:
# Does the chasing advantage change from season to season?
# claim:Some seasons favour the chasing team much more than others.
# h0-The chase win rate is the same in every season, and the differences are luck.
# h1=The chase win rate is not the same in every season..
df=pd.read_sql("""SELECT season_year,
    SUM(chase_won)     AS won,
    SUM(1 - chase_won) AS lost
FROM  v_match_totals
GROUP BY season_year;""",con)
df

,season_year,won,lost
0,2008,36,22
1,2009,30,26
2,2010,28,31
3,2011,40,32
4,2012,40,34
5,2013,37,37
6,2014,37,22
7,2015,24,32
8,2016,41,19
9,2017,32,26


In [29]:
chi2,p,dof,expected=stats.chi2_contingency(df[['won','lost']])
print(round(chi2, 2))
print(round(p, 4))
print(p<0.05)

20.34
0.3141
False


In [30]:
df=pd.read_sql("""SELECT bowler, bowler_type,
    6.0*SUM(bowler_runs)/SUM(is_legal) AS economy,
    SUM(is_legal) AS balls
FROM v_ball
WHERE TRIM(COALESCE(bowler_type,'')) <> ''
GROUP BY bowler, bowler_type
HAVING balls >= 500;""",con)
df

,bowler,bowler_type,economy,balls
0,A Kumble,Legbreak Googly,6.578238,965
1,A Mishra,Legbreak,7.377633,3371
2,A Nehra,Left arm Medium fast,7.845912,1908
3,A Nortje,Right arm Fast,9.085714,1120
4,A Symonds,"Right arm Medium, Right arm Offbreak",7.673624,527
...,...,...,...,...
144,YK Pathan,Right arm Offbreak,7.401918,1147
145,YS Chahal,Legbreak Googly,8.027912,3941
146,Yash Dayal,Left arm Medium fast,9.575201,871
147,Yuvraj Singh,Slow Left arm Orthodox,7.436133,869


In [31]:
styles = df['bowler_type'].value_counts()
styles = styles[styles >= 8].index         # keep styles with 8+ bowlers
groups = [df[df['bowler_type'] == s].economy for s in styles]
F, p = stats.f_oneway(*groups)
print(round(F, 2))
print(p)
print(len(groups))
print(p<0.05)

6.99
5.916709274557887e-07
8
True
